<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

# Introduction

Hi, my name is **Iain Derrington** — I'm an **FAE and Platforms Engineer** within the **ADEF group at Analog Devices (ADI)**.  
I've been with ADI for just over two years.

My background is fairly broad:
- Spent **10–15 years** as an **electronic design engineer** in various industries, including **ADEF** and **optical communications**
- Then moved into **firmware development** for aproximates **10 years**
- Recently (2023) shifted into the **sales and applications** side

I've also run two businesses:
- An electronics design consultancy called **Hot Solder**, for about 5 years
- And (randomly!) a **sign design and manufacturing company** called **K&I Design**, also for around 5 years (part-time)


### Outside of Work

When I’m not working:
- I love playing board games — I currently own around **80 games**
- I enjoy mountain biking 🚴‍♂️ (these days, the bike has a battery…)
- And I play guitar 🎸 (enthusiastically, if not expertly)


### About Today’s Session

Today's presentation is delivered using **Jupyter Notebooks** — which not only allow me to share and structure information clearly, but also provide a powerful, interactive environment to **develop and run live scripts** during the session.


### Links

- [CN-0566 Product Page](https://www.analog.com/en/resources/reference-designs/circuits-from-the-lab/cn0566.html#rd-overview)
- [CN-0566 Wiki Page](https://wiki.analog.com/resources/eval/user-guides/circuits-from-the-lab/cn0566)
- [Jon Kraft Youtube Channel](https://www.youtube.com/@jonkraft)
- [LinkedIn Profile](https://www.linkedin.com/in/iderrington)
### Contact Details

Iain.Derrington@analog.com <br>
Simon.Walkin@analog.com

This notebook introduces the fundamentals of RADAR, explores the major RADAR types used in modern systems, and then focuses on Frequency Modulated Continuous Wave (FMCW) RADAR — the technique we will later implement on the PlutoSDR and Phaser array.


# What is RADAR

RAdio Detection and Ranging (RADAR) refers to systems that use radio waves to measure / determine:

- Range (distance)
- Velocity (speed)
- Angle (azimuth / elevation)
- Target Characteristics

The core idea:

1. Transmit electromagnetic waveforms.
2. Receive echos reflected from objects
3. Process the returned signal.

RADAR is used in many applications including  — aviation, automotive, defence, drones, satellites, maritime navigation, weather, and industrial sensing.

Broadly speaking there are two types of active RADAR, pulsed radar and Continuous Wave (CW) RADAR. This workshop focuses on CW RADAR, FMCW to be specific. To understand why we might want to use FMCW RADAR, we will quickly review pulsed RADAR in the next section.
Before we do as reminder lets take a look at the RADAR range equation, it applies to both CW and pulsed RADAR:

$ \begin{align}
\large  P_r = \frac{\textcolor{green}{P_t} G_t G_r \lambda^2 \sigma}{(4 \pi)^2 \textcolor{red}{r^4}}     \quad (1)
\end{align}
$

 - $P_r$ = Rx Power (W)
 - $P_t$ = Tx Power (W)
 - $G_t$ =  Tx Antenna gain
 - $G_r$ =  Rx Antenna gain
 - $\lambda$ = Wavelength (m) of RADAR signal
 - $\sigma$ = RADAR cross section in $m^2$
 - $r$ = range (m)

The only variables we can easily change are the antenna gains in beam forming application, the $\lambda $ and the $P_t$. Practically, changing $\lambda$ is going to be difficult.

Notice the $P_r$ is directly proportional to $\frac{1}{r^4}$. So the distance away has a huge impact on the power level at the RADAR receiver.

All practical RADAR systems will need a minimum $P_r$ to achieve a signal to noise ratio that is suitable for the application. 

## Basics of Pulse Radar

Pulse RADAR, as the name suggests, transmits short pulses of RF energy. Each electromagnetic (EM) pulse radiates away from the antenna, strikes a target, and a portion of the energy is reflected back toward the radar. The RADAR system measures the round-trip time of the pulse:

<div style="text-align: center;">
<video controls src = "resources/RadarWavePropagationScene.mp4" width=600>
</video>
</div>

The distance of the object can be calculated as:

$
\Large   d = \frac{c\cdot t}{2}  \quad (2)
$

Where: 

- $d$ =  distance in metres
- $c$ = Speed of light
- $t$ = time in seconds



### Pulse Length and Range Resolution

The more power that is reflected back to the receiver, the better the signal-to-noise ratio (SNR). One way to increase the received energy is to transmit longer pulses. However, increasing the pulse length has a downside: it makes it harder to distinguish between two targets that are close together in range.

The ability to distinguish two closely spaced targets is known as range resolution:
<div style="text-align: center;">
<video controls src = "resources/RangeResolutionMovingBlocksScene.mp4" width=600> </video>
</div>

As shown, the range resolution is directly proportional to the pulse width. Shorter pulses result in better range resolution:

$
\Large  \Delta R = \frac{c  \tau}{2} \quad(3)
$

where $\tau$ is the pulse width (seconds).

If the pulse width is $1,\mu s$:

$
\Large  \Delta R = \frac{3 \times 10 ^ 8 \cdot 1 \times 10 ^{-6}}{2} =  150m
$

This means that two targets must be at least 150 m apart to be resolved as separate objects when using a 1 µs pulse.




### Bandwidth and Power Trade-offs

It is worth noting that the bandwidth requirement of a pulsed radar is inversely proportional to pulse width:

Shorter pulses → larger bandwidth

Longer pulses → smaller bandwidth

The challenge with very short pulses is that they contain less energy, so achieving a good SNR requires very high peak transmit power.

From the radar equation (Equation 1), the received power $P_r$ scales as:

$
\Large P_r \propto \frac{1}{R^4}
$

This severe $R^4$ dependence means that long-range detection demands enormous transmit power.

Increasing pulse length increases transmitted energy, but as demonstrated above, long pulses degrade range resolution. This fundamental trade-off motivates techniques such as pulse compression, which we will return to later.

### What are Pulsed RADARs good for

* **Very long range capability**
  * Can transmit extremely high peak power (kW–MW).
  * Excellent for long-range surveillance and early-warning radar.
  * Classic choice for air-defence, weather radar, and ATC.
* **Simple and unambiguous range measurement**
  * No frequency mixing required.
  * No range-Doppler coupling (unlike FMCW).
* **Mature and well-understood architecture**
  * Decades of operational experience.
  * Robust signal processing.
  * Easier to reason about conceptually.

### What Pulsed RADARs Are Not So Good At

* **High peak power hardware**
  * Larger, heavier, more expensive systems.
  * Harder to miniaturise.
* **Lower average power efficiency**
  * High peak power but low duty cycle.
  * Average transmitted energy may be lower than FMCW for the same thermal budget.
* **Blind range**
  * Radar cannot receive while transmitting.
* **Range–resolution trade-off**
  * Short pulses → good resolution but low energy.
  * Long pulses → poor resolution unless you add pulse compression (chirped pulses).

## The BASICS of CW RADAR

We have briefly looked at pulsed RADAR and seen why it is useful in many applications, but also why it may not be suitable in others. An alternative approach is Continuous-Wave (CW) RADAR, where instead of transmitting short pulses of energy, the transmitter emits a continuous RF waveform:

The main limitation of basic CW RADAR is that it is not well suited to range measurement. However, it is very effective at measuring target velocity using the Doppler effect. If a target is moving towards or away from the RADAR, the frequency of the reflected signal will increase or decrease accordingly:

<div style="text-align: center;">
<video controls src = "resources/CWRadarWaveDopplerScene_FullRX.mp4" width=600> </video>
</div>


CW RADAR is therefore well suited to velocity measurement, but much less useful for measuring range. This naturally raises the question:

**What do we need to change in order to resolve range?**



### Introducing Frequency Modulation

If we frequency-modulate the carrier, we can exploit relatively simple mathematics to extract range information. This leads directly to Frequency-Modulated Continuous-Wave (FMCW) RADAR.

We will begin with a linear frequency ramp transmitted by the RADAR and assume a stationary target, allowing us to focus on the underlying mathematics.

### Linear Chirp Signal

A linear chirp is a sinusoid whose frequency starts at $f_0$ and increases linearly with time. The instantaneous transmit frequency can be written as:

$\Large  f_{tx}(t) = f_0 + k\cdot t \quad (4)$

To generate a sinusoidal signal, we require a phase term. Since frequency is the time derivative of phase, the signal can be written as

$\Large y(t) = sin (2 \pi\cdot\phi(t)) \quad (5)$

Phase is the integral of frequency, thus integrating (4) with respect to time yields:

$
\Large \phi(t) = f_0 t + \frac{1}{2} k t^2 \quad (6)
$

Substituting (6) into (5) yields the expression for a linear chirp:

$ \Large y(t) =  sin (2 \pi\cdot f_0t + \frac{1}{2}kt^2) $

Where:

- $f_0$ is the start frequency
- $k = \frac{B}{T}$ is the chirp slope (Hz/s)
- $B$ is the sweep bandwidth
- $T$ is the chirp duration

> Note: The chirp slope 𝑘
> k has units of Hz/s, so multiplying by time gives frequency.

Let’s now code this up — it will be useful later when we begin processing real FMCW signals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def gen_linear_chirp(t, f_start, f_end, T):
    """
    Linear chirp:
      f(t) = f_start + k*t,  where k = (f_end - f_start)/T
      phase(t) = 2π( f_start*t + 0.5*k*t^2 )
      x(t) = sin(phase(t))

    Returns:
      x: waveform
      f_inst: instantaneous frequency (linear ramp)
    """
    k = (f_end - f_start) / T
    phase = 2*np.pi*(f_start*t + 0.5*k*t**2)
    x = np.sin(phase)
    f_inst = f_start + k*t
    return x, f_inst

def plot_chirp(f_start=1_000, f_end=10_000, T_ms=5.0, N=2000):
    # Convert ms to seconds
    T = T_ms * 1e-3

    # Safety: avoid weird cases
    if T <= 0:
        return

    t = np.linspace(0, T, int(N), endpoint=False)

    x, f_inst = gen_linear_chirp(t, f_start, f_end, T)

    plt.figure(figsize=(10, 4))
    ax1 = plt.gca()
    ax1.plot(t, x)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(t, f_inst)
    ax2.set_ylabel("Instantaneous Frequency (Hz)")

    k = (f_end - f_start) / T  # Hz/s
    plt.title(f"Linear Chirp  |  f0={f_start:.0f} Hz  f1={f_end:.0f} Hz  T={T_ms:.2f} ms  k={k/1e6:.3f} MHz/s")
    plt.show()

# Sliders
f_start_slider = widgets.IntSlider(value=1000, min=10, max=50_000, step=10, description="f_start (Hz)")
f_end_slider   = widgets.IntSlider(value=10_000, min=10, max=50_000, step=10, description="f_end (Hz)")
T_slider       = widgets.FloatSlider(value=5.0, min=0.1, max=50.0, step=0.1, description="T (ms)")
N_slider       = widgets.IntSlider(value=2000, min=200, max=20000, step=200, description="N")

# Keep end >= start automatically (nice UX)
def clamp_end(*args):
    if f_end_slider.value < f_start_slider.value:
        f_end_slider.value = f_start_slider.value

f_start_slider.observe(clamp_end, names="value")

ui = widgets.VBox([
    widgets.HBox([f_start_slider, f_end_slider]),
    widgets.HBox([T_slider, N_slider]),
])

out = widgets.interactive_output(
    plot_chirp,
    {
        "f_start": f_start_slider,
        "f_end": f_end_slider,
        "T_ms": T_slider,
        "N": N_slider,
    },
)

display(ui, out)


### Why Use a Linear Chirp?

Why would we want to create a linear chirp, and how does this solve the range-finding problem in CW RADAR?

Consider a target at range $R$. The reflected signal returns to the RADAR after a round-trip delay (see [Eq. 3](#Pulse-Length-and-Range-Resolution)):

$
\Large \tau = \frac{2R}{c} \quad(7)
$

The received signal is therefore a delayed copy of the transmitted signal:

$
\Large f_{rx}(t) = f_{tx}(t - \tau)  \quad(8)
$

This means that, at any given instant, the received signal has the frequency that was transmitted $\tau$ seconds earlier. The difference between the transmitted and received frequencies is known as the beat frequency:

$
\Large f_b = f_{tx}(t) - f_{rx}(t)  \quad(9)
$

Substituting (8) in to (9)

$
\Large f_b = f_{tx} - f_{tx}(t-\tau)
$

If the transmit frequency is a linear chirp,

$
\Large f_{tx}(t) = f_0 + k t
$

then substituting gives:

$
\Large
\begin{aligned}
f_b        &= f_0 + kt - \bigl(f_0 + k(t - \tau)\bigr) \\
           &= \cancel{f_0} + \cancel{kt} - \cancel{f_0} -  \cancel{kt} + k\tau \\
           &= k \tau \quad(10)
\end{aligned}
$

Using the expression for delay (7) and substuting in (10), this becomes:

$
\Large f_b = k \frac{2R}{c}
$

Rearranging for range yields the fundamental FMCW ranging equation:

$
\Large R = \frac{c}{2k} \cdot f_b  \quad (11)
$

### Key Insight

By using a linear frequency ramp, a time delay is converted into a constant frequency difference. FMCW RADAR therefore replaces short time measurements with precise frequency measurements.

In effect, FMCW uses frequency instead of time as the ranging ruler, allowing long, low-power transmissions while still achieving fine range resolution.

The example below illustrates how the difference between the transmitted and received signals produces a beat frequency that is directly proportional to target range.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# -----------------------------
# Parameters
# -----------------------------
c = 299_792_458.0   # m/s

f_start  = 1e3      # Hz
f_end    = 10e3     # Hz
T        = 5e-3     # chirp duration (s)
n_chirps = 3

fs = 400_000        # sample rate for drawing waveforms
k = (f_end - f_start) / T  # chirp slope (Hz/s)

t_total = n_chirps * T
N = int(t_total * fs)
t = np.linspace(0, t_total, N, endpoint=False)

# -----------------------------
# Signal generators
# -----------------------------
def tx_stream(t):
    """Continuous stream of identical up-chirps. Returns tx(t), f_tx(t)."""
    t_mod = np.mod(t, T)
    f_tx = f_start + k * t_mod
    phase = 2*np.pi*(f_start*t_mod + 0.5*k*t_mod**2)
    tx = np.exp(1j * phase)  # complex linear chirp
    return tx, f_tx

def rx_from_delay(t, tau):
    """
    RX is a delayed copy of TX by tau seconds.
    For t < tau: no echo yet (rx=0, f_rx=nan).
    """
    rx = np.zeros_like(t, dtype=complex)
    f_rx = np.full_like(t, np.nan, dtype=float)

    t_del = t - tau
    valid = t_del >= 0
    if np.any(valid):
        rx_valid, f_valid = tx_stream(t_del[valid])
        rx[valid] = rx_valid
        f_rx[valid] = f_valid

    return rx, f_rx

tx, f_tx = tx_stream(t)

# -----------------------------
# Beat frequency estimation
# -----------------------------
def estimate_beat_and_range(t, beat, tau):
    """
    Estimate beat frequency from the complex beat signal:
        beat(t) = rx(t) * conj(tx(t)) = exp(j * phi_b(t))

    We estimate instantaneous frequency via phase difference:
        f_inst ≈ (1/(2π)) * d/dt (unwrap(angle(beat)))

    Then take a robust median within a stable region.
    """
    # Guard against chirp reset edge (modulo discontinuity)
    t_mod = np.mod(t, T)
    guard = 0.05 * T
    stable = (t_mod > guard) & (t_mod < (T - guard)) & (t >= tau)

    if np.count_nonzero(stable) < 100:
        return np.nan, np.nan

    # Phase -> instantaneous frequency
    phi = np.unwrap(np.angle(beat))
    dphi = np.diff(phi)
    f_inst = (fs / (2*np.pi)) * dphi  # Hz, length N-1

    stable_f = stable[1:]  # align with diff
    if np.count_nonzero(stable_f) < 50:
        return np.nan, np.nan

    fb_hat = float(np.median(f_inst[stable_f]))
    R_hat = (c / (2 * k)) * fb_hat
    return fb_hat, R_hat

# -----------------------------
# Plot
# -----------------------------
def plot_fmcw(tau_ms=0.8, zoom_ms=5.0):
    tau = tau_ms * 1e-3
    rx, f_rx = rx_from_delay(t, tau)

    # Beat (mixed) signal at baseband
    beat = rx * np.conj(tx)

    fb_hat, R_hat = estimate_beat_and_range(t, beat, tau)

    # ---- Plot ----
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True)

    # 1) Time-domain: show REAL part only (keeps it visually simple)
    ax0.plot(t*1e3, np.real(tx), label="TX (real)")
    ax0.plot(t*1e3, np.real(rx), label=f"RX (real), delay τ={tau_ms:.2f} ms")
    ax0.set_ylabel("Amplitude")
    ax0.set_title("TX and delayed RX (time domain)")
    ax0.grid(True)
    ax0.legend(loc="upper right")

    # 2) Instantaneous frequency
    ax1.plot(t*1e3, f_tx, label="f_tx(t)")
    ax1.plot(t*1e3, f_rx, label="f_rx(t)=f_tx(t-τ)")
    ax1.set_ylabel("Frequency (Hz)")
    ax1.set_xlabel("Time (ms)")
    ax1.set_title("Instantaneous frequency (3 chirps)")
    ax1.grid(True)
    ax1.legend(loc="upper right")

    # ---- Clean info box ----
    if np.isfinite(fb_hat):
        info = (
            f"Delay:  τ = {tau_ms:0.3f} ms\n"
            f"Beat:   f_b = {fb_hat:0.1f} Hz\n"
            f"Range:  R = {R_hat:0.3f} m"
        )
    else:
        info = (
            f"Delay:  τ = {tau_ms:0.3f} ms\n"
            "Beat/Range: not enough overlap"
        )

    ax1.text(
        0.02, 0.98, info,
        transform=ax1.transAxes,
        ha="left", va="top",
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", alpha=0.9)
    )

    # ---- Zoom ----
    zoom_ms = max(0.1, float(zoom_ms))
    ax0.set_xlim(0, zoom_ms)

    plt.tight_layout()
    plt.show()

max_tau_ms = (n_chirps*T - 0.1e-3) * 1e3

interact(
    plot_fmcw,
    tau_ms=FloatSlider(value=0.8, min=0.0, max=max_tau_ms, step=0.05, description="τ (ms)"),
    zoom_ms=FloatSlider(value=5.0, min=0.5, max=n_chirps*T*1e3, step=0.25, description="Zoom (ms)")
);


You may notice the beat frequency is negative. This is because the received chirp is delayed; for an up-chirp, the delayed signal corresponds to a slightly lower frequency than the current transmit frequency, so after mixing the result appears as a negative beat frequency. Range depends on the magnitude of the beat frequency, not its sign.

This is a big step: we can now derive range for a stationary target. Later, we’ll extend this to moving targets and see how to extract both range and velocity. But before we do that, let’s take what we’ve learned so far and make it work on real hardware.

### Range Resolution

We've shown in (7) the range of a target is determined by the beat frequency. Distinguishing two close targets will be realated to the receive systems abillity to distinguish the two beat frequencies:

Two close targets have a beat-frequency separation:

$ \Large f_b \approx k \frac{2}{c} \Delta R $

Intuitively, the steeper the chirp slope (k) the greater $f_b$ will be and the easier it will be to measue the differing frequencies. Another consideration is the duration of the chirp... remember from DSP theory, the longer a signal is observed for the better the FFT resoltuion.

It is shown (Appendix)the range resolution of a FMCW can be aproximated as follows:

$ \Large \Delta R = \frac{c}{2B}  \quad (12)$ 

Lets put some numbers in here to see the impact. The typical BW for the [AD9361](https://www.analog.com/en/products/ad9361.html) is aprx 20MHz. Thus:

$ \Large \Delta R = \frac{c}{2 \times  20e6}   = 7.5m$ 

For the [AD9084](https://www.analog.com/en/products/ad9084.html) the iBW can be up to 10GHz. Thus:

$ \Large \Delta R = \frac{c}{2 \times  10e9}   = 0.015m = 1.5cm!!$

# Summary

Pulse RADAR measures range directly using time-of-flight, but faces a fundamental trade-off between range resolution, bandwidth, and peak transmit power.

CW RADAR transmits continuously and is excellent for measuring target velocity via Doppler, but cannot measure range on its own.

FMCW RADAR combines continuous transmission with frequency modulation, converting time delay into a beat frequency and enabling accurate range measurement with low peak power.

Together, these architectures illustrate how modern RADAR systems trade time, frequency, power, and signal processing to meet different application requirements.

# Appendix

## FMCW Range Res Math
FFT bins are set by the lenght or time over which the FFT is run. The FFT bin width can be aproximated to:

$ \Large {fft}_{bin} \approx \frac{1}{T} $

So to distinguish between two beat frequencies, the frequencies must be greate or equal to the FFT bin:

$ \Large  f_b \geq \frac{1}{T}$

$
\Large f_b = k \frac{2\Delta R}{c}  \geq \frac{1}{T}
$

Re arrange for R:

$
\Large \Delta R \geq \frac{c}{2kT}
$

Subtitute in chirp slope $k = \frac{B}{T}$

$
\Large 
\begin{aligned}
\Delta R &= \frac{c}{2\frac{B}{\cancel{T}} \cancel{T}} \\
         &= \frac{c}{2B} 
\end{aligned}
$